1. ensure the correct output with julia
2. construct the DCOPF latex tutorial
3. check the PU in the formulation, since the 30bus is infeasible

In [12]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from scipy.stats import multivariate_normal
import numpy as np
from tqdm import tqdm
file=".\\old\\DCOPF-main\\DCOPF-main\\excel_outputs\\pglib_opf_case73_ieee_rts.xlsx"
mpc_data = pd.read_excel(file, sheet_name=['baseMVA', 'bus', 'gen', 'gencost', 'branch'])

In [13]:
def omega_sample(buses, Pd, sigma_scaling=0.03, nsamples=1000):
    stdomega = {b: sigma_scaling*Pd[b] for b in buses}
    # nonzeroindices = [i for i in range(len(stdomega)) if stdomega[i] > 1e-5]
    # mean = np.zeros(len(nonzeroindices))
    # cov = np.diag(list(map(stdomega.__getitem__, nonzeroindices)))**2
    mean = {b: 0 for b in buses}
    # cov = {b: stdomega[b]**2 for b in buses}
    omega_samples = {(b,s): mean[b] + stdomega[b]*np.random.randn() if stdomega[b] > 1e-5 else 0
                     for b in buses for s in range(nsamples)}
    omega_samples = {(b,s): omega_samples[b,s] if omega_samples[b,s] > 1e-5 else 0
                     for b in buses for s in range(nsamples)}
    # omega = multivariate_normal.rvs(mean=mean, cov=cov, size=nsamples)
    # omega_samples = np.zeros((len(buses), nsamples))
    # omega_samples[nonzeroindices] = omega.T if omega.ndim == 2 else omega[:, np.newaxis]
    return omega_samples

In [14]:
# Create gurobipy Model
model = gp.Model("DCOPF")
# === Sets ===
buses = mpc_data['bus']['bus_i'].tolist()
buses_index_busID = dict(zip(mpc_data['bus'].index,mpc_data['bus']['bus_i']))
buses_busID_index = dict(zip(mpc_data['bus']['bus_i'],mpc_data['bus'].index))
gens = mpc_data['gen']['gen_ID'].tolist()
branches = mpc_data['branch'].index.tolist()
branches_ftbus = dict(zip(branches,mpc_data['branch'][['bus_i', 'bus_j']].values))
# === Parameters ===
# Generator cost coefficients (all costs are incorporated)
c = {}
for i in mpc_data['gencost']["gen_ID"]:
    c[i] = [mpc_data['gencost']['c2'][i-1],mpc_data['gencost']['c1'][i-1],mpc_data['gencost']['c0'][i-1]]
# Bus power demand (MW)
Pd = dict(zip(mpc_data['bus']['bus_i'], mpc_data['bus']['Pd']))
# shunt conductance (MW demanded at V = 1.0 p.u.)
Gs = dict(zip(mpc_data['bus']['bus_i'], mpc_data['bus']['Gs']))
# Generator capacity limits (MW)
Pmax = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['Pmax']))    
Pmin = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['Pmin']))
# Transmission line limits (MW)
Pmax_line = dict(zip(branches, mpc_data['branch']['rateA']))
Pmin_line = dict(zip(branches, -mpc_data['branch']['rateA']))
# Line susceptance (1/X), assuming per unit values
# B = dict(zip(branches, 1/(mpc_data['branch']['x'])))
B = dict(zip(branches, mpc_data['branch']['x']/(mpc_data['branch']['x']**2+mpc_data['branch']['r']**2)))
# Generator bus assignment
gen_bus = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['bus_i']))
# === Variables ===
Pg = model.addVars(gens, lb=Pmin, ub=Pmax, vtype=GRB.CONTINUOUS, name="Pg")
theta = model.addVars(buses, lb=-100, ub=100, vtype=GRB.CONTINUOUS, name="theta")
P_flow = model.addVars(branches, lb=Pmin_line, ub=Pmax_line, vtype=GRB.CONTINUOUS, name="P_flow")
omega = model.addVars(buses, lb=0, ub=0, vtype=GRB.CONTINUOUS, name="omega")
# === Objective Function (Minimize Generation Cost, all costs are incorporated) ===
model.setObjective(gp.quicksum(c[i][0]*Pg[i] + c[i][1]*Pg[i] + c[i][2] for i in gens), GRB.MINIMIZE)
# === Power Balance Constraints ===
for b in buses:
    expr = (gp.quicksum(Pg[i] for i in gen_bus if gen_bus[i] == b)
            + gp.quicksum(P_flow[l] for l, ft in branches_ftbus.items() if ft[1] == b)
            - gp.quicksum(P_flow[l] for l, ft in branches_ftbus.items() if ft[0] == b))
    model.addConstr(expr == Pd[b] + Gs[b] + omega[b] , name=f"power_balance_{b}")
# === Line Flow Constraints (DC Power Flow) ===
for l in branches:
    model.addConstr(P_flow[l] == B[l]*(theta[branches_ftbus[l][0]] - theta[branches_ftbus[l][1]]), name=f"line_flow_{l}")
# === Reference Bus Constraint (Slack Bus) ===
ref_bus_index = mpc_data['bus'][mpc_data['bus']['type'] == 3].index[0]
model.addConstr(theta[buses_index_busID[ref_bus_index]] == 0, name="theta_ref") # buses_index_busID[ref_bus_index] gets the ref bus
# === Solve Model Using Gurobi ===
model.Params.OptimalityTol = 1e-8 # Higher precision
model.setParam('OutputFlag', 0) # suppress the output
model.optimize() 

Set parameter OptimalityTol to value 1e-08


In [15]:
for v in model.getVars():
    print('%s %g' % (v.VarName, v.X))

Pg[1] 16
Pg[2] 16
Pg[3] 76
Pg[4] 76
Pg[5] 16
Pg[6] 16
Pg[7] 76
Pg[8] 76
Pg[9] 100
Pg[10] 100
Pg[11] 100
Pg[12] 69
Pg[13] 69
Pg[14] 69
Pg[15] 0
Pg[16] 2.4
Pg[17] 2.4
Pg[18] 2.4
Pg[19] 2.4
Pg[20] 2.4
Pg[21] 155
Pg[22] 155
Pg[23] 400
Pg[24] 400
Pg[25] 50
Pg[26] 50
Pg[27] 50
Pg[28] 50
Pg[29] 50
Pg[30] 50
Pg[31] 155
Pg[32] 155
Pg[33] 350
Pg[34] 16
Pg[35] 16
Pg[36] 76
Pg[37] 76
Pg[38] 16
Pg[39] 16
Pg[40] 76
Pg[41] 76
Pg[42] 25
Pg[43] 100
Pg[44] 79
Pg[45] 69
Pg[46] 69
Pg[47] 69
Pg[48] 0
Pg[49] 2.4
Pg[50] 2.4
Pg[51] 2.4
Pg[52] 2.4
Pg[53] 2.4
Pg[54] 155
Pg[55] 155
Pg[56] 400
Pg[57] 400
Pg[58] 50
Pg[59] 50
Pg[60] 50
Pg[61] 50
Pg[62] 50
Pg[63] 50
Pg[64] 155
Pg[65] 155
Pg[66] 350
Pg[67] 16
Pg[68] 16
Pg[69] 76
Pg[70] 76
Pg[71] 16
Pg[72] 16
Pg[73] 76
Pg[74] 76
Pg[75] 25
Pg[76] 25
Pg[77] 25
Pg[78] 69
Pg[79] 69
Pg[80] 69
Pg[81] 0
Pg[82] 2.4
Pg[83] 2.4
Pg[84] 2.4
Pg[85] 2.4
Pg[86] 2.4
Pg[87] 155
Pg[88] 155
Pg[89] 400
Pg[90] 400
Pg[91] 50
Pg[92] 50
Pg[93] 50
Pg[94] 50
Pg[95] 50
Pg[96] 50
Pg[97] 155
Pg[9

In [4]:
class OPF_Scenarios:
    def __init__(self,noptimal, scenarios, solutions, cbases, rbases, whichbasis, whichscenario):
        self.noptimal = noptimal
        self.scenarios = scenarios
        self.solutions = solutions
        self.cbases = cbases
        self.rbases = rbases
        self.whichbasis = whichbasis
        self.whichscenario = whichscenario
        
def OPFScenarios(model, omega, omega_samples, nsamples):
    status = [None] * nsamples
    soln_p = np.zeros((nsamples, len(gens)))
    cbases = {}
    rbases = {}
    sample_omega = {}
    noptimal = 0
    for s in tqdm(range(nsamples)):
        # model.setAttr("LB", omega, omega_samples[:,s])
        # model.setAttr("UB", omega, omega_samples[:,s])
        for b in omega:
            omega[b].lb = 0
            # omega[b].lb = omega_samples[b,s]
            omega[b].ub = 0
            # omega[b].ub = omega_samples[b,s]
#         m.model.setParam('OutputFlag', 0) # suppress the output
        # print("Variables:")
        # for v in model.getVars():
        #     if (v.varName[:5] == "omega") and (v.ub!=0)and (v.lb!=0):
        #         print(f"{v.varName}: lb={v.lb}, ub={v.ub}, obj={v.Obj}")
        # if s >10:
        #     break
        model.optimize()
        status[s] = model.status
        if status[s] == GRB.OPTIMAL:
            soln_p[s,:] = list(model.getAttr('x', Pg).values())
            noptimal += 1
            cbasis = tuple(model.getAttr('Vbasis', model.getVars()))
            rbasis = tuple(model.getAttr('Cbasis', model.getConstrs()))
            cbases[cbasis] = cbases.get(cbasis, [])
            rbases[rbasis] = rbases.get(rbasis, [])
            cbases[cbasis].append(noptimal)
            rbases[rbasis].append(noptimal)
            sample_omega[s] = {b:omega_samples[b,s] for b in buses}
    assert noptimal == sum(1 for stat in status if stat == 2), 'Mismatch in optimal scenario count'
    sample_p = soln_p[np.array(status)==GRB.OPTIMAL,:]
    # sample_omega = omega_samples[:, np.array(status)==GRB.OPTIMAL]
    colbases = list(cbases.keys())
    rowbases = list(rbases.keys())
    whichcol = dict(zip(colbases, range(len(colbases))))
    whichrow = dict(zip(rowbases, range(len(rowbases))))
    whichbasis = np.zeros((noptimal, 2), dtype=int)
    for ckey in cbases.keys():
        whichbasis[cbases.get(ckey)[-1]-1,0] = whichcol[ckey]
    for rkey in rbases.keys():
        whichbasis[rbases.get(rkey)[-1]-1,1] = whichrow[rkey]
    whichscenario = {}
    for i in range(noptimal):
        basiskey = (whichbasis[i,0], whichbasis[i,1])
        whichscenario[basiskey] = whichscenario.get(basiskey, [])
        whichscenario[basiskey].append(i)
    return OPF_Scenarios(noptimal, sample_omega, sample_p, colbases, rowbases, whichbasis, whichscenario)

In [5]:
np.random.seed(0)
nsamples=1
omega_samples = omega_sample(buses, Pd, sigma_scaling=0.03, nsamples=nsamples)
scenarios = OPFScenarios(model, omega, omega_samples,nsamples)
scenarios.noptimal

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 35.74it/s]


1

In [8]:
len(scenarios.solutions[0])

49